# Launch forecast: how it works and how to improve it

Compares the current `ai_forecast` setup with an `ai_forecast` v2 variant, a global **LightGBM model tracked in MLflow**, and two baselines, using a rolling-origin backtest.

**Framing**
* **Series:** one per `client_id` (vendor), weekly, Monday-based (`DATE_TRUNC('week', launch_date)`)
* **Target:** `y` = number of products launching that week, from `silver.product_launches`
* **Horizon:** 12 weeks; **cutoff** = last complete Monday-week ≤ today
* **History vs scheduled:** weeks after the cutoff hold *announced* launches (known in advance), so they are kept apart as **scheduled**, not used as history
* **Covariates:** calendar only (week of year, month), which is known in advance
* **Metric:** MASE (primary, scaled by in-sample naive m=1 because history is under two yearly seasons), MAE, RMSE, sMAPE, and 90% interval coverage

**Caveat:** at the time of writing, NIVEA's history is 100% synthetic test data, and Henkel has no history, only one scheduled week. Results show how methods handle the synthetic pattern, not real launch behaviour.

In [0]:
%pip install -q lightgbm
dbutils.library.restartPython()

In [0]:
import datetime as dt, time, warnings
import numpy as np, pandas as pd
import lightgbm as lgb, mlflow, matplotlib.pyplot as plt
from mlflow.models import infer_signature
from databricks.sdk import WorkspaceClient
warnings.filterwarnings("ignore")

CATALOG, WAREHOUSE_ID, H = "patricia_aws_serverless_catalog", "4471bd9074d14824", 12
EXPERIMENT = "/Users/patricia.ferreiro@databricks.com/launch-forecast-comparison"
w = WorkspaceClient()

def run_wh(sql: str) -> pd.DataFrame:
    """Run SQL on the SQL warehouse (ai_forecast is not enabled on serverless notebooks)."""
    r = w.statement_execution.execute_statement(warehouse_id=WAREHOUSE_ID, statement=sql, wait_timeout="50s")
    while r.status.state.value in ("PENDING", "RUNNING"):
        time.sleep(2); r = w.statement_execution.get_statement(r.statement_id)
    if r.status.state.value != "SUCCEEDED":
        raise RuntimeError(r.status.error.message)
    cols = [c.name for c in r.manifest.schema.columns]
    rows, res = [], r.result
    while res is not None:
        rows += res.data_array or []
        res = w.statement_execution.get_statement_result_chunk_n(r.statement_id, res.next_chunk_index) if res.next_chunk_index else None
    return pd.DataFrame(rows, columns=cols)

In [0]:
weekly = spark.sql(f"""
  SELECT client_id, CAST(DATE_TRUNC('week', launch_date) AS DATE) AS ds, COUNT(*) AS y,
         SUM(CASE WHEN source_file LIKE 'synthetic://%' THEN 1 ELSE 0 END) AS y_synthetic
  FROM {CATALOG}.silver.product_launches
  WHERE launch_date IS NOT NULL AND client_id IS NOT NULL
  GROUP BY 1, 2""").toPandas()
weekly["ds"] = pd.to_datetime(weekly["ds"])

TODAY = pd.Timestamp(dt.date.today())
CUTOFF = TODAY - pd.Timedelta(days=TODAY.weekday()) - pd.Timedelta(weeks=1)  # last complete Monday-week
observed, scheduled = weekly[weekly.ds <= CUTOFF], weekly[weekly.ds > CUTOFF].sort_values(["client_id", "ds"])

def zero_fill(df):
    out = []
    for c, g in df.groupby("client_id"):
        idx = pd.date_range(g.ds.min(), CUTOFF, freq="W-MON")
        s = g.set_index("ds").reindex(idx)
        out.append(pd.DataFrame({"client_id": c, "ds": idx, "y": s["y"].fillna(0).astype(float).values,
                                 "y_synthetic": s["y_synthetic"].fillna(0).values}))
    return pd.concat(out, ignore_index=True)

series = zero_fill(observed)
summary = series.groupby("client_id").agg(weeks=("y", "size"), first=("ds", "min"), last=("ds", "max"),
                                          zero_weeks=("y", lambda s: int((s == 0).sum())), mean=("y", "mean"))
summary["synthetic_share"] = (series.groupby("client_id")["y_synthetic"].sum() / series.groupby("client_id")["y"].sum()).round(3)
print("today", TODAY.date(), "| cutoff week", CUTOFF.date())
print("clients with no history before the cutoff:", sorted(set(weekly.client_id) - set(series.client_id)))
display(summary.reset_index())
display(scheduled[["client_id", "ds", "y"]])

In [0]:
fig, ax = plt.subplots(figsize=(11, 3.5))
for c, g in series.groupby("client_id"):
    ax.plot(g.ds, g.y, marker=".", label=f"{c} history")
for c, g in scheduled.groupby("client_id"):
    ax.scatter(g.ds, g.y, marker="x", s=60, label=f"{c} scheduled")
ax.axvline(CUTOFF, color="grey", lw=.8); ax.set_ylabel("launches / week"); ax.legend(fontsize=8); ax.set_title("Weekly launches per client")
display(fig)

In [0]:
# Issues in the current output: one horizon for all clients (from the latest week of any client) and NULL rows for short histories
display(spark.sql(f"""
  SELECT client_id, COUNT(*) AS points, MIN(ds) AS first_ds, MAX(ds) AS last_ds,
         SUM(CASE WHEN metric_forecast IS NULL THEN 1 ELSE 0 END) AS null_forecasts,
         ROUND(AVG(metric_forecast), 2) AS avg_forecast, ROUND(AVG(metric_upper - metric_lower), 2) AS avg_interval_width
  FROM {CATALOG}.gold.launch_forecasts GROUP BY client_id"""))

In [0]:
HOLDOUT_ORIGIN = CUTOFF - pd.Timedelta(weeks=H)
FOLDS = [(f"backtest_{k}", HOLDOUT_ORIGIN - pd.Timedelta(weeks=k * H)) for k in (3, 2, 1)] + [("holdout", HOLDOUT_ORIGIN)]
bt_clients = [c for c, g in series.groupby("client_id") if (g.ds <= FOLDS[0][1]).sum() >= 8]
print("folds:", [(n, o.date()) for n, o in FOLDS], "| clients in the backtest:", bt_clients)

def target_weeks(origin):
    return pd.date_range(origin + pd.Timedelta(weeks=1), periods=H, freq="W-MON")

def m_seasonal_naive(train, origin):
    out = []
    for c, g in train.groupby("client_id"):
        hist = g.set_index("ds")["y"]
        out += [(c, t, hist.get(t - pd.Timedelta(weeks=52), hist.iloc[-1]), np.nan, np.nan) for t in target_weeks(origin)]  # else last value
    return pd.DataFrame(out, columns=["client_id", "ds", "yhat", "lo", "hi"])

def m_mean8(train, origin):
    out = []
    for c, g in train.groupby("client_id"):
        v = g.sort_values("ds")["y"].tail(8).mean()
        out += [(c, t, v, np.nan, np.nan) for t in target_weeks(origin)]
    return pd.DataFrame(out, columns=["client_id", "ds", "yhat", "lo", "hi"])

def values_cte(df):  # W-MON needs a TIMESTAMP time column
    rows = ",\n".join(f"('{r.client_id}', TIMESTAMP'{r.ds.date()} 00:00:00', {float(r.y)})" for r in df.itertuples())
    return f"t AS (SELECT * FROM VALUES {rows} AS v(client_id, ds, y))"

V2 = "positive_only => true, holiday_region => 'DE', frequency => 'W-MON', prediction_interval_width => 0.90"

def m_ai(variant):
    def f(train, origin):
        data = train[train.y > 0] if variant == "current" else train  # the current pipeline only sees non-empty weeks
        args = ["TABLE(t)", f"horizon => DATE'{(origin + pd.Timedelta(weeks=H, days=1)).date()}'", "time_col => 'ds'",
                "value_col => 'y'", "group_col => 'client_id'"]
        args += (["prediction_interval_width => 0.90", "frequency => 'W-MON'", "parameters => '{\"global_floor\": 0}'"]
                 if variant == "current" else ["version => '2'", V2])
        r = run_wh(f"WITH {values_cte(data)} SELECT * FROM ai_forecast({', '.join(args)})")
        r["ds"] = pd.to_datetime(r["ds"]).dt.tz_localize(None)
        r = r.rename(columns={"y_forecast": "yhat", "y_lower": "lo", "y_upper": "hi"})
        return r.astype({"yhat": float, "lo": float, "hi": float})[["client_id", "ds", "yhat", "lo", "hi"]]
    return f

def score(pred, train, actual):
    df, rows = actual.merge(pred, on=["client_id", "ds"], how="left"), []
    for c, g in df.groupby("client_id"):
        scale = np.mean(np.abs(np.diff(train[train.client_id == c].sort_values("ds")["y"].values)))  # in-sample naive (m=1)
        e, denom = g.y - g.yhat, (np.abs(g.y) + np.abs(g.yhat))
        rows.append(dict(client_id=c, mase=np.mean(np.abs(e)) / scale, mae=np.mean(np.abs(e)), rmse=np.sqrt(np.mean(e ** 2)),
                         smape=np.mean(np.where(denom == 0, 0, 2 * np.abs(e) / denom)) * 100,
                         coverage=np.mean((g.y >= g.lo) & (g.y <= g.hi)) if g.lo.notna().all() else np.nan,
                         n_missing_pred=int(g.yhat.isna().sum())))
    return rows

In [0]:
LAGS = [12, 13, 14, 26, 52]  # every lag >= horizon, so future rows only need known history (no leakage)
FEATURES = [f"lag_{l}" for l in LAGS] + ["roll_mean_12_19", "weekofyear", "month", "client"]
LGB_PARAMS = dict(n_estimators=200, learning_rate=0.05, num_leaves=7, min_child_samples=5, verbose=-1, random_state=42)

def make_features(frame):
    frame, hist = frame[["client_id", "ds", "y"]].copy(), frame.dropna(subset=["y"])
    for l in LAGS:
        lagged = hist[["client_id", "ds", "y"]].rename(columns={"y": f"lag_{l}"})
        lagged["ds"] = lagged["ds"] + pd.Timedelta(weeks=l)
        frame = frame.merge(lagged, on=["client_id", "ds"], how="left")
    roll = []
    for c, g in hist.groupby("client_id"):
        r = g.set_index("ds")["y"].asfreq("W-MON").rolling(8, min_periods=4).mean()
        r.index = r.index + pd.Timedelta(weeks=12)  # mean of weeks t-19..t-12
        roll.append(pd.DataFrame({"client_id": c, "ds": r.index, "roll_mean_12_19": r.values}))
    frame = frame.merge(pd.concat(roll), on=["client_id", "ds"], how="left")
    frame["weekofyear"], frame["month"] = frame.ds.dt.isocalendar().week.astype(int), frame.ds.dt.month
    frame["client"] = frame.client_id.astype("category")
    return frame

def fit_lgb(train):
    f = make_features(train).dropna(subset=["lag_12"])
    models = {q: lgb.LGBMRegressor(objective="quantile", alpha=q, **LGB_PARAMS).fit(f[FEATURES], f["y"]) for q in (0.05, 0.95)}
    models["mean"] = lgb.LGBMRegressor(objective="regression", **LGB_PARAMS).fit(f[FEATURES], f["y"])
    return models, len(f)

def predict_lgb(models, history, future):
    frame = make_features(pd.concat([history[["client_id", "ds", "y"]], future.assign(y=np.nan)], ignore_index=True))
    rows = frame[frame.y.isna()].copy()
    rows["client"] = pd.Categorical(rows.client_id, categories=sorted(history.client_id.unique()))
    yhat = np.clip(models["mean"].predict(rows[FEATURES]), 0, None)
    lo, hi = (np.clip(models[q].predict(rows[FEATURES]), 0, None) for q in (0.05, 0.95))
    out = rows[["client_id", "ds"]].assign(yhat=yhat, lo=np.minimum(lo, yhat), hi=np.maximum(hi, yhat))
    return future[["client_id", "ds"]].merge(out, on=["client_id", "ds"], how="left")

def m_lgb(train, origin):
    future = pd.DataFrame([(c, t) for c in train.client_id.unique() for t in target_weeks(origin)], columns=["client_id", "ds"])
    return predict_lgb(fit_lgb(train)[0], train, future)

In [0]:
METHODS = {"seasonal_naive": m_seasonal_naive, "mean_8w": m_mean8, "ai_forecast_current": m_ai("current"),
           "ai_forecast_v2": m_ai("v2"), "lightgbm_global": m_lgb}
# A separate 'zero-filled ai_forecast' run is only different when history has empty weeks
print("empty weeks in backtest history:", int((series[series.client_id.isin(bt_clients)].y == 0).sum()))

results, preds = [], {}
for fold, origin in FOLDS:
    train = series[(series.ds <= origin) & series.client_id.isin(bt_clients)]
    actual = series[series.ds.isin(target_weeks(origin)) & series.client_id.isin(bt_clients)][["client_id", "ds", "y"]]
    assert len(actual) == H * len(bt_clients)
    for name, fn in METHODS.items():
        preds[(name, fold)] = p = fn(train, origin)
        results += [dict(method=name, fold=fold, origin=origin.date(), **r) for r in score(p, train, actual)]
allres = pd.DataFrame(results)
display(allres.round(3))

In [0]:
hold = allres[allres.fold == "holdout"].set_index("method")[["mase", "mae", "rmse", "smape", "coverage"]].add_prefix("holdout_")
bt = allres[allres.fold != "holdout"].groupby("method")[["mase", "mae", "coverage"]].mean().add_prefix("backtest_mean_")
comparison = hold.join(bt).sort_values("holdout_mase")
display(comparison.round(3).reset_index())

# Rule: lowest holdout MASE, interval coverage 60-100%, and it must beat seasonal naive; otherwise the best baseline
sn = comparison.loc["seasonal_naive", "holdout_mase"]
eligible = comparison[(comparison.holdout_mase < sn) & comparison.holdout_coverage.between(0.6, 1.0)]
WINNER = eligible.index[0] if len(eligible) else comparison.loc[["seasonal_naive", "mean_8w"], "holdout_mase"].idxmin()
print("selected method:", WINNER)

In [0]:
def plot_holdout(method, ax):
    hist = series[(series.client_id == bt_clients[0]) & (series.ds > HOLDOUT_ORIGIN - pd.Timedelta(weeks=26))]
    ax.plot(hist.ds, hist.y, color="#1b3139", marker=".", label="actual")
    p = preds[(method, "holdout")]
    ax.plot(p.ds, p.yhat, color="#ff3621", ls="--", label="forecast")
    if p.lo.notna().all(): ax.fill_between(p.ds, p.lo, p.hi, color="#ff3621", alpha=.15, label="90% interval")
    ax.axvline(HOLDOUT_ORIGIN, color="grey", lw=.8); ax.set_title(method, fontsize=10); ax.legend(fontsize=7)

top = list(comparison.index[:3])
fig, axes = plt.subplots(1, len(top), figsize=(15, 3.5), sharey=True)
for m, ax in zip(top, axes): plot_holdout(m, ax)
fig.tight_layout(); display(fig)

In [0]:
class LaunchForecaster(mlflow.pyfunc.PythonModel):
    """Takes future rows (client_id, ds = Monday of the week); returns forecast, forecast_lower, forecast_upper."""
    def __init__(self, models, history):
        self.models = models
        self.history = history[["client_id", "ds", "y"]].sort_values("ds").groupby("client_id").tail(max(LAGS) + 8)
    def predict(self, context, model_input, params=None):
        future = model_input[["client_id", "ds"]].copy(); future["ds"] = pd.to_datetime(future["ds"])
        p = predict_lgb(self.models, self.history, future)
        return future.merge(p.rename(columns={"yhat": "forecast", "lo": "forecast_lower", "hi": "forecast_upper"}),
                            on=["client_id", "ds"], how="left")

mlflow.set_experiment(EXPERIMENT)
run_ids, MODEL_URI = {}, None
for method in allres.method.unique():
    m = allres[allres.method == method]
    with mlflow.start_run(run_name=method) as run:
        mlflow.set_tags({"project": "product-launch-forecast", "method": method})
        mlflow.log_params({"horizon_weeks": H, "cutoff": str(CUTOFF.date()), "folds": len(FOLDS), "mase_scale": "in-sample naive (m=1)"})
        if method == "ai_forecast_v2": mlflow.log_param("ai_forecast_args", V2)
        for r in m.itertuples():
            for k in ("mase", "mae", "rmse", "smape", "coverage"):
                if pd.notna(getattr(r, k)): mlflow.log_metric(f"{r.fold}_{k}", float(getattr(r, k)))
        for k in ("mase", "mae", "rmse", "smape", "coverage"):
            v = m[m.fold != "holdout"][k]
            if v.notna().any(): mlflow.log_metric(f"backtest_mean_{k}", float(v.mean()))
        fig, ax = plt.subplots(figsize=(9, 3.5)); plot_holdout(method, ax); mlflow.log_figure(fig, "holdout_forecast_vs_actual.png"); plt.close(fig)
        if method == "lightgbm_global":
            full = series[series.client_id.isin(bt_clients)]
            final_models, n_final = fit_lgb(full)  # refit on all history up to the cutoff
            model = LaunchForecaster(final_models, full)
            future = pd.DataFrame([(c, t) for c in bt_clients for t in target_weeks(CUTOFF)], columns=["client_id", "ds"])
            mlflow.log_params({**{f"lgb_{k}": v for k, v in LGB_PARAMS.items()}, "lags": str(LAGS), "n_final_fit": n_final})
            MODEL_URI = mlflow.pyfunc.log_model(name="model", python_model=model, signature=infer_signature(future, model.predict(None, future)),
                                                input_example=future.head(5), extra_pip_requirements=[f"lightgbm=={lgb.__version__}"]).model_uri
        run_ids[method] = run.info.run_id

# The logged model must forecast every future row
fc = mlflow.pyfunc.load_model(MODEL_URI).predict(future)
assert len(fc) == len(future) and fc["forecast"].notna().all(), "logged model cannot forecast future rows"
print("model:", MODEL_URI); display(fc)

In [0]:
acc = allres.assign(horizon_weeks=H, evaluated_at=pd.Timestamp.utcnow().tz_localize(None), cutoff=CUTOFF.date(),
                    mlflow_run_id=allres.method.map(run_ids), selected=allres.method.eq(WINNER))
acc = acc[["client_id", "method", "fold", "origin", "cutoff", "horizon_weeks", "mase", "mae", "rmse", "smape", "coverage",
           "selected", "mlflow_run_id", "evaluated_at"]]
spark.createDataFrame(acc).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.launch_forecast_accuracy")
display(spark.table(f"{CATALOG}.gold.launch_forecast_accuracy"))

## Conclusion (run of 2026-11-21, cutoff week 2026-11-09)

| Method | Holdout MASE | Holdout MAE | Holdout 90% coverage | Backtest mean MASE |
| --- | --- | --- | --- | --- |
| **LightGBM (global, MLflow)** | **2.03** | **1.84** | 75% | **1.34** |
| ai_forecast (current settings) | 2.45 | 2.22 | 75% | 2.49 |
| ai_forecast v2 (positive_only, DE holidays) | 2.59 | 2.34 | 33% | 1.73 |
| Seasonal naive (52 weeks) | 2.67 | 2.42 | – | 2.70 |
| Mean of last 8 weeks | 2.71 | 2.46 | – | 2.21 |

* **Selected by the rule: LightGBM.** It has the lowest holdout MASE, believable coverage, and beats seasonal naive; it is also best on the backtest mean.
* **Best `ai_forecast` variant: the current settings.** v2 is better on the backtest mean but its 90% interval covered only 33% of holdout weeks, so the SQL pipeline keeps the current settings.
* **Pipeline fixes regardless of method:** history only up to the cutoff (announced launches are *scheduled*, not history), zero-filled weeks, the same 12-week horizon for every client, a baseline for clients with fewer than 8 weeks of history (no more NULL rows), and scheduled launches as a floor for the forecast.
* **Not switched yet:** moving the job to the LightGBM model needs Unity Catalog registration and a Python job task.
* **Caveat:** only NIVEA could be backtested and its history is 100% synthetic, so rerun this notebook once real history accumulates before relying on the ranking.